# NLPA 114-2 Hugging Face Tutorial

In [2]:
!pip install rouge==1.0.1

In [3]:
import os
import torch
import jieba
import wandb
from transformers import AutoTokenizer
from transformers import GPT2LMHeadModel
from datasets import load_dataset
from tqdm.auto import tqdm
from rouge import Rouge

In [4]:
class LCSTSDataset(torch.utils.data.Dataset):
    def __init__(self, raw_data) -> None:
        super().__init__()
        self.data = raw_data
        # To prevent out-of-vocabulary tokens from being transformed into [UNK]
        self.token_replacement = [
            ["：", ":"],
            ["，", ","],
            ["“", '"'],
            ["”", '"'],
            ["？", "?"],
            ["……", "..."],
            ["！", "!"],
        ]

    def __getitem__(self, index):
        d = dict(self.data[index])  # 🔹 複製一份，不動到原始資料
        # Substitute some full-width punctuations with half-width ones
        for k in d:
            for tok in self.token_replacement:
                d[k] = d[k].replace(tok[0], tok[1])
        return d

    def __len__(self):
        return len(self.data)

In [5]:
# `pad_token_id`=`tokenizer.eos_token_id`:
# For each batch, first finished sentences should have <|endoftext|> rather than [PAD] at the end.
# Check more details from the following link.
# https://github.com/huggingface/transformers/blob/b880508440f43f80e35a78ccd2a32f3bde91cb23/src/transformers/generation_utils.py#L1248-L1251

# `max_new_tokens`: If you don’t set max_new_tokens,
# Hugging Face will also count the input tokens!

def do_evaluate(
    tokenizer,
    model,
    validation_loader,
    rouge_metric,
    inner_check=False,
    max_eval_samples=100,
):
    model.eval()  # 切換到 eval mode（關閉 dropout）
    pbar = tqdm(validation_loader)
    pbar.set_description(f"Evaluating")

    predictions = []
    references = []
    articles = []

    with torch.no_grad():  # 不記錄梯度，省 VRAM + 加速
        for ground_truth, inputs in pbar:
            output = [
                s.split("[SEP]")[1].replace(" ", "").split("<|endoftext|>")[0]
                for s in tokenizer.batch_decode(
                    model.generate(
                        **inputs,
                        max_new_tokens=200, # Maximum number of tokens to generate
                        pad_token_id=tokenizer.eos_token_id,
                    )
                )
            ]
            targets = [
                s.split("[SEP]")[1].replace(" ", "").replace("<|endoftext|>", "")
                for s in tokenizer.batch_decode(ground_truth["input_ids"])
            ]

            raw_articles = [
                s.replace(" ", "")  # 中文 decode 後會有空格，去掉
                for s in tokenizer.batch_decode(inputs["input_ids"], skip_special_tokens=True)
            ]

            assert len(output) == len(targets)
            # 處理 batch 內任何位置的空字串
            output = [o if o.strip() else " " for o in output]
            # We use jieba to perform word-level evaluations with ROUGE
            predictions.extend([" ".join(jieba.lcut(o)) for o in output])
            references.extend([" ".join(jieba.lcut(t)) for t in targets])
            articles.extend(raw_articles)
            # 用 predictions 的長度判斷是否停止 (前提：inner_check == True)
            if inner_check and len(predictions) >= max_eval_samples:
                break

    score = rouge_metric.get_scores(predictions, references, avg=True)

    model.train()  # 切換回 train mode
    return score, predictions, references, articles

In [6]:
def collate_fn(batch):
    complete_text = [
        f"[CLS]{example['text']}[SEP]{example['summary']}<|endoftext|>"
        for example in batch
    ]
    complete_text = tokenizer(
        complete_text,
        padding=True,
        truncation=True,
        return_tensors="pt",
        add_special_tokens=False,
    )
    # Set label padding tokens to -100 for loss masking
    labels = torch.where(
        condition=complete_text.input_ids != tokenizer.pad_token_id,
        input=complete_text.input_ids,
        other=-100,
    )
    complete_text["labels"] = labels
    complete_text = {k: complete_text[k].to(device) for k in complete_text}

    infer_text = [example["text"] for example in batch]
    infer_text = tokenizer(
        infer_text,
        padding=True,
        truncation=True,
        return_tensors="pt",
    )
    infer_text = {k: infer_text[k].to(device) for k in infer_text}
    return complete_text, infer_text

In [7]:
TRAIN_BATCH_SIZE = 32
VAL_BATCH_SIZE = 64 # During evaluation, we don't pad the input.
NUM_EPOCHS = 3
LR = 1e-5
MODEL_NAME = "uer/gpt2-chinese-cluecorpussmall"

In [8]:
SAVED_DIR = "saved_models"
os.makedirs(SAVED_DIR, exist_ok=True)  # 建立資料夾

In [10]:
# wandb (Weights & Biases) 初始化
wandb.init(
    project="nlp-class-gpt2-summarization-test",  # 同 project 下所有 run 會被歸類
    name=f"bs{TRAIN_BATCH_SIZE}_lr{LR}",     # 這次 run 的名字
    config={                                 # 自動記錄超參數！
        "model": MODEL_NAME,
        "train_batch_size": TRAIN_BATCH_SIZE,
        "val_batch_size": VAL_BATCH_SIZE,
        "num_epochs": NUM_EPOCHS,
        "learning_rate": LR,
    },
)

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: yingjia-lin (yingjia-lin-cgu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [11]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    padding_side="left", # Use left padding for GPT2
)
# You can set your device id instead of cuda:0
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/577 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/217 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [12]:
# Sometimes checking the Hugging Face dataset is slow,
# it will be faster if we transform the dataset object into a list using .to_list().

raw_train = load_dataset(
    "hugcyp/LCSTS",
    split="train",
    cache_dir="./cache/",
).to_list()

raw_val = load_dataset(
    "hugcyp/LCSTS",
    split="validation",
    cache_dir="./cache/"
).to_list()

train.jsonl:   0%|          | 0.00/903M [00:00<?, ?B/s]

valid.jsonl: 0.00B [00:00, ?B/s]

test_public.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/2400591 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/8685 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/725 [00:00<?, ? examples/s]

In [13]:
train_set = LCSTSDataset(raw_train)
val_set = LCSTSDataset(raw_val)

In [14]:
train_loader = torch.utils.data.DataLoader(
    train_set,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
)
val_loader = torch.utils.data.DataLoader(
    val_set,
    batch_size=VAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
)

In [15]:
# `resize_token_embeddings`:
# Increasing the size will add newly initialized vectors at the end.

model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)
tokenizer.add_special_tokens({"eos_token": "<|endoftext|>"}) # Add a new eos token
model.resize_token_embeddings(len(tokenizer))
model = model.to(device)

pytorch_model.bin:   0%|          | 0.00/421M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/421M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: uer/gpt2-chinese-cluecorpussmall
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivari

In [16]:
# Set up the optimizer and the evaluation metric

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
rouge_metric = Rouge()

In [ ]:
step_i = 0
for epoch in range(NUM_EPOCHS):
    pbar = tqdm(train_loader)
    pbar.set_description(f"Training epoch [{epoch+1}/{NUM_EPOCHS}]")
    for inputs, _ in pbar:
        optimizer.zero_grad()
        loss = model(**inputs).loss
        loss.backward()
        optimizer.step()

        # 取得 loss 數值
        loss_value = loss.item()

        pbar.set_postfix(loss=loss_value)
        # Log the loss to TensorBoard. 每 10 步記錄一次
        if step_i % 10 == 0:
            wandb.log({"loss/train": loss_value}, step=step_i)

        if step_i % 5000 == 0 and step_i != 0: # Evaluate every 1000 steps
            score, pres, refs, articles = do_evaluate(
                tokenizer=tokenizer,
                model=model,
                validation_loader=val_loader,
                rouge_metric=rouge_metric,
                inner_check=True,
            )
            print(f"Rouge scores on step {step_i} of epoch {epoch}:", score)
            print("Articles:", articles[:5]) # Check the first 5 articles
            print("Predictions:", pres[:5]) # Check the first 5 predictions
            print("References:", refs[:5])  # Check the first 5 references

            # Log ROUGE scores to Weights & Biases (wandb)
            wandb.log({
                "rouge-1/val": score["rouge-1"]["f"],
                "rouge-2/val": score["rouge-2"]["f"],
                "rouge-l/val": score["rouge-l"]["f"],  # 順便補上 ROUGE-L
            }, step=step_i)
        step_i += 1

    score, pres, refs, articles = do_evaluate(
        tokenizer=tokenizer,
        model=model,
        validation_loader=val_loader,
        rouge_metric=rouge_metric,
    )
    model.save_pretrained(f"{SAVED_DIR}/ep{epoch}")
    tokenizer.save_pretrained(f"{SAVED_DIR}/ep{epoch}")

    # 把部分預測結果和對應的文章、摘要一起記錄到 wandb，方便檢視
    table = wandb.Table(columns=["Article", "Prediction", "Reference"])
    for art, pred, ref in zip(articles[:20], pres[:20], refs[:20]):
        table.add_data(art, pred, ref)
    wandb.log({f"epoch_{epoch}_samples": table})

# 訓練結束後，結束 wandb run
wandb.finish()

  0%|          | 0/75019 [00:00<?, ?it/s]

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
